**This problem set is due Wednesday, September 16, 2026 at 11:59 pm. Please plan ahead and submit your work on time.**

### Problem Set 01: Uninformed Search

In this Problem Set you will implement Breadth First Search and Depth First Search and use them to solve the [8 Puzzle Problem](https://en.wikipedia.org/wiki/15_puzzle).

0. [Credit for Contributors (required)](#contributors)
1. [State Representation in the 8 Puzzle Problem (35 points)](#state_representation)
    1. [Successor Function (25 points)](#state_expansion)
    2. [Completing the `PuzzleProblem` class (10 points)](#puzzle_problem_class)
2. [Simple Search (52 points)](#simple_search)
    1. [Breadth First Search (27 points)](#bfs_implementation)
    2. [Depth First Search (25 points)](#dfs_implementation)
3. [Differences Between BFS and DFS (8 points)](#ai_component)
    1. [Using an LLM to Find a Differentiating Example [AI-enabled] (4 points)](#llm_differentiating_example)
    2. [Visualize the Domain [AI-enabled] (4 points)](#visualize_domain)
4. [Time Spent on Pset (5 points)](#time_spent)

**100 points** total for Problem Set 1

## <a name="contributors"></a> Credit for Contributors

List the various students, lecture notes, or online resouces that helped you complete this problem set:

Ex: I worked with Bob on the cat activity planning problem.

<div class="alert alert-info">
Write your answer in the cell below this one.
</div>

--> *(double click on this cell to delete this text and type your answer here)*

Import the modules needed for this exercise (make sure you execute the cell below by clicking on it and pressing Shift-Enter)

**Do not import any other modules**

In [1]:
%load_ext autoreload
%autoreload 2
from search_classes import SearchNode, Path
from principles_of_autonomy.grader import Grader
from principles_of_autonomy.notebook_tests.pset_1 import TestPSet1
import networkx
import matplotlib.pyplot as plt

## <a name="state_representation"></a>Problem 1: State Representation in the 8 Puzzle Problem

The puzzle consists of a 3x3 grid with 8 numbered tiles and a missing tile. The objective is to slide the tiles around until all the numbered tiles are ordered and the missing tile stays at the lower right cell of the grid.

<img src="puzzle8.png"/>

To make things simple, we are giving you a possible state representation for the 8-puzzle problem.

We'll represent a given state of the puzzle by a tuple of three internal tuples. Each internal tuple represents a row of the puzzle. The missing tile is represented by $0$.

For example, the puzzle state below:

<img src="example_state.png"/>

is represented by `((1, 2, 3), (8, 0, 4), (7, 6, 5))`.

Below, we are giving you some code to print a puzzle state:


In [7]:
def print_state(state):
    print("+"+ "-"*5+"+")
    for l in state:
        print("|"+ " ".join([str(el) if el!=0 else " " for el in l]) +"|")
    print("+"+ "-"*5+"+")

example_state = ((1, 2, 3), (8, 0, 4), (7, 6, 5))

print("%s state represents puzzle state: " % (example_state,))
print_state(example_state)

((1, 2, 3), (8, 0, 4), (7, 6, 5)) state represents puzzle state: 
+-----+
|1 2 3|
|8   4|
|7 6 5|
+-----+


### <a name="state_expansion"></a>1.A Successor Function (25 points)

In order to find a solution to the search problem, we need to define the states we can reach from a given state. This corresponds to the possible moves of the missing tile (at most up, down, left and right).

Implement the function `expand_state(state)` that returns a `list` of the states that can be reached from the given `state`.

For example, for state `((0, 1, 3), (4, 2, 5), (7, 8, 6))`, the function `expand_state` should return the following list (two moves are feasible):

```
[((4, 1, 3), (0, 2, 5), (7, 8, 6)), ((1, 0, 3), (4, 2, 5), (7, 8, 6))]
```

The neighbour states of state:

```
+-----+
|  1 3|
|4 2 5|
|7 8 6|
+-----+
```

are:

```
+-----+
|4 1 3|
|  2 5|
|7 8 6|
+-----+
***
+-----+
|1   3|
|4 2 5|
|7 8 6|
+-----+
```

<div class="alert alert-info">
Implement the function `expand_state(state)` below.
</div>


In [126]:
from typing import Any


def expand_state(state):
    print_state(state)
    state_list = list(state)
    # identiffy zero position
    for i in range(0,3):
        for j in range(0,3):
            if (state_list[i][j] == 0):
                zero_pos_i = i
                zero_pos_j = j
    new_state_left = []
    new_state_right = []
    new_state_top = []
    new_state_bottom = []

    # left
    if zero_pos_j > 0:
        current_left = state_list[zero_pos_i][zero_pos_j - 1]
        
        stable_rows = list(set([zero_pos_i]) ^ set([0,1,2]))
        unstable_row = zero_pos_i
        for i in range(0,3):
            if i in stable_rows:
                new_state_left.append(tuple(state_list[i]))
            else:
                new_row = list(state_list[i])
                new_row[zero_pos_j] = current_left
                new_row[zero_pos_j -1] = 0
                new_state_left.append(tuple(new_row))
    
    # right
    if zero_pos_j < 2:
        current_right = state_list[zero_pos_i][zero_pos_j + 1]
        
        stable_rows = list(set([zero_pos_i]) ^ set([0,1,2]))
        unstable_row = zero_pos_i
        for i in range(0,3):
            if i in stable_rows:
                new_state_right.append(tuple(state_list[i]))
            else:
                new_row = list(state_list[i])
                new_row[zero_pos_j] = current_right
                new_row[zero_pos_j + 1] = 0
                new_state_right.append(tuple(new_row))

    # top
    if zero_pos_i > 0:
        current_top = state_list[zero_pos_i - 1][zero_pos_j]
        
        stable_rows = list(set([zero_pos_i, zero_pos_i -1]) ^ set([0,1,2]))
        unstable_rows = [zero_pos_i - 1, zero_pos_i]

        for i in range(0,3):
            if i in stable_rows:
                new_state_top.append(tuple(state_list[i]))
            else:
                if i == zero_pos_i:
                    new_row = list(state_list[i])
                    new_row[zero_pos_j] = current_top
                    new_state_top.append(tuple(new_row))
                else:
                    new_row = list(state_list[i])
                    new_row[zero_pos_j] = 0
                    new_state_top.append(tuple(new_row))
                

    # bottom
    if zero_pos_i < 2:
        current_bottom = state_list[zero_pos_i + 1][zero_pos_j]
        
        stable_rows = list(set([zero_pos_i, zero_pos_i + 1]) ^ set([0,1,2]))

        unstable_rows = [zero_pos_i + 1, zero_pos_i]

        for i in range(0,3):
            if i in stable_rows:
                new_state_bottom.append(tuple(state_list[i]))
            else:
                if i == zero_pos_i:
                    new_row = list(state_list[i])
                    new_row[zero_pos_j] = current_bottom
                    new_state_bottom.append(tuple(new_row))
                else:
                    new_row = list(state_list[i])
                    new_row[zero_pos_j] = 0
                    new_state_bottom.append(tuple(new_row))




    final_list = []
    if new_state_left:
        final_list.append(tuple(new_state_left))
    if new_state_right:
        final_list.append(tuple(new_state_right))
    if new_state_top:
        final_list.append(tuple(new_state_top))
    if new_state_bottom:
        final_list.append(tuple(new_state_bottom))
    return final_list


expand_state(((0, 1, 3), (4, 2, 5), (7, 8, 6)))

+-----+
|  1 3|
|4 2 5|
|7 8 6|
+-----+


[((1, 0, 3), (4, 2, 5), (7, 8, 6)), ((4, 1, 3), (0, 2, 5), (7, 8, 6))]

In [127]:
Grader.run_single_test_inline(TestPSet1, "test_1_check_expanded_states", locals())

.
----------------------------------------------------------------------
Ran 1 test in 0.002s

OK


+-----+
|  1 3|
|4 2 5|
|7 8 6|
+-----+
+-----+
|1 2 3|
|8   4|
|7 6 5|
+-----+


### <a name="puzzle_problem_class"></a> 1.B Completing the `PuzzleProblem` class (10 points)

We are giving you the class `SearchNode` defined in `search_classes.py`. This class represents a search node in the search tree. You can create a `SearchNode` by giving it the state it represents and its `SearchNode` parent (or None if it's the root element in the tree). Below is an example of the `SearchNode` class being used:

In [128]:
# Execute this example code
root_node = SearchNode(((0, 1, 3), (4, 2, 5), (7, 8, 6)), parent_node=None)
children_node = SearchNode(((4, 1, 3), (0, 2, 5), (7, 8, 6)),
                            parent_node=root_node)
print("Root node: %s" % root_node)
print("Children node: %s" % children_node)

Root node: <SearchNode: state: ((0, 1, 3), (4, 2, 5), (7, 8, 6)), parent: None>
Children node: <SearchNode: state: ((4, 1, 3), (0, 2, 5), (7, 8, 6)), parent: <SearchNode: state: ((0, 1, 3), (4, 2, 5), (7, 8, 6)), parent: None>>


We also give you the `Path` class, that takes a `SearchNode` and computes the state path from the initial state in the root of the tree to the state of the given `SearchNode`:

In [129]:
# Execute this example code
example_path = Path(children_node)
print("Path of %d states is: %s" % (len(example_path.path), example_path.path))

Path of 2 states is: [((0, 1, 3), (4, 2, 5), (7, 8, 6)), ((4, 1, 3), (0, 2, 5), (7, 8, 6))]


Implement the function `expand_node(self, search_node)` inside the `PuzzleProblem` class. The function should return a `list` of the successor SearchNodes that can be reached from the given `search_node`.


<div class="alert alert-warning">
You will want to look at the `SearchNode` and `Path` definitions in the included `search_classes.py` file, as you will need to know what useful properties you can use for the next questions.
</div>

<div class="alert alert-info">
Implement the function `expand_node(self, search_node)` below.
</div>

In [140]:
class PuzzleProblem(object):
    """Class that represents the puzzle search problem."""
    def __init__(self, start, goal):
        self.start = start
        self.goal = goal
    def test_goal(self, state):
        return self.goal == state
    def expand_node(self, search_node):
        """Return a list of SearchNodes, having the correct state and parent node."""
        new_nodes = expand_state(search_node._state)
        new_search_nodes = []
        for i in range(len(new_nodes)):
            tmp_node = SearchNode(new_nodes[i],
                            parent_node=search_node)
            new_search_nodes.append(tmp_node)
            

        return new_search_nodes
        
        

In [141]:
Grader.run_single_test_inline(TestPSet1, "test_2_puzzle_problem_expanded_nodes", locals())

.


----------------------------------------------------------------------
Ran 1 test in 0.002s

OK


+-----+
|1 2 3|
|8   4|
|7 6 5|
+-----+


## <a name="simple_search"></a>Problem 2: Simple Search

Now you will implement Simple Search, as seen in class, to solve the 8 Puzzle Problem. 

### <a name="bfs_implementation"></a>2.A Breadth First Search (27 points)

First, you'll implement *Breadth First Search*.

Implement the function `breadth_first_search(search_problem)` that takes an instance of the `PuzzleProblem` class that we defined above and returns a tuple of three elements, in the following order:

1. If BFS finds a solution, an instance of the `Path` class containing that solution. If it doesn't, it should return `None` as the first element of the tuple.
2. The number of visited nodes
3. The maximum size of the queue

You should use a **visited list**, as otherwise the number of explored states in this problem will be large.

<div class="alert alert-info">
Implement `breadth_first_search(search_problem)` below.
</div>

In [ ]:
from re import search


def breadth_first_search(search_problem):
    """This function should take a PuzzleProblem instance and return a 3 element tuple as described above."""
    root_node = SearchNode(search_problem.start,parent_node=None)
    new_children = search_problem.expand_node(root_node)

    visited_list = []

    for child in new_children:
        print(child.__eq__)
        if child._state == search_problem.goal:
            print("yes")
        else:
            print("no")
            visited_list.append(child._state)


    




# Solve the 8 Puzzle Problem from state:
# +-----+
# |  1 3|
# |4 2 5|
# |7 8 6|
# +-----+
# Don't modify this cell (contents will be overwritten by autograder)
# If you want to experiment with other states, try adding cells below.
# You can try with state: ((1, 8, 2), (0, 4, 3), (7, 6, 5)) for example.
# Remember that not all states have a solution. Try ((8, 1, 2), (0, 4, 3), (7, 6, 5)), for example.
# Be ready to wait, though!
start_state = ((0, 1, 3), (4, 2, 5), (7, 8, 6))
# start_state = ((1, 8, 2), (0, 4, 3), (7, 6, 5))
goal_state = ((1,2,3),(4,5,6),(7,8,0))
problem = PuzzleProblem(start_state, goal_state)

breadth_first_search(problem)
# if sol:    
#     print(
#         "Solution found!\n%d states in the solution (%d moves)\n"
#         "%d states explored.\n%d maximum queue"
#         % (len(sol.path), len(sol.path) - 1, num_visited, max_q)
#     )
#     print("Solution: ")
#     for s in sol.path:
#         print_state(s)
#         print("\n**\n")
# else:
    # print("No solution after exploring %d states with max q of %d" %(num_visited, max_q))

+-----+
|  1 3|
|4 2 5|
|7 8 6|
+-----+
<bound method SearchNode.__eq__ of <SearchNode: state: ((1, 0, 3), (4, 2, 5), (7, 8, 6)), parent: <SearchNode: state: ((0, 1, 3), (4, 2, 5), (7, 8, 6)), parent: None>>>
no
<bound method SearchNode.__eq__ of <SearchNode: state: ((4, 1, 3), (0, 2, 5), (7, 8, 6)), parent: <SearchNode: state: ((0, 1, 3), (4, 2, 5), (7, 8, 6)), parent: None>>>
no


### Solve the Puzzle Problem using BFS

Let's use your Breadth First Search implementation to solve the 8 Puzzle Problem.
Execute the cell below. If your BFS implementation is correct, you should see the solution printed below.

In [105]:
# Solve the 8 Puzzle Problem from state:
# +-----+
# |  1 3|
# |4 2 5|
# |7 8 6|
# +-----+
# Don't modify this cell (contents will be overwritten by autograder)
# If you want to experiment with other states, try adding cells below.
# You can try with state: ((1, 8, 2), (0, 4, 3), (7, 6, 5)) for example.
# Remember that not all states have a solution. Try ((8, 1, 2), (0, 4, 3), (7, 6, 5)), for example.
# Be ready to wait, though!
start_state = ((0, 1, 3), (4, 2, 5), (7, 8, 6))
# start_state = ((1, 8, 2), (0, 4, 3), (7, 6, 5))
goal_state = ((1,2,3),(4,5,6),(7,8,0))
problem = PuzzleProblem(start_state, goal_state)

sol, num_visited, max_q = breadth_first_search(problem)
if sol:    
    print(
        "Solution found!\n%d states in the solution (%d moves)\n"
        "%d states explored.\n%d maximum queue"
        % (len(sol.path), len(sol.path) - 1, num_visited, max_q)
    )
    print("Solution: ")
    for s in sol.path:
        print_state(s)
        print("\n**\n")
else:
    print("No solution after exploring %d states with max q of %d" %(num_visited, max_q))

NotImplementedError: 

In [ ]:
Grader.run_single_test_inline(TestPSet1, "test_3_bfs", locals())

### <a name="dfs_implementation"></a>2.B Depth First Search (25 points)

Next, you'll implement *Depth First Search*.

Implement the function `depth_first_search(search_problem)` that takes an instance of the `PuzzleProblem` class that we defined above and also returns a tuple of three elements, in the following order:

1. If DFS finds a solution, an instance of the `Path` class containing that solution. If it doesn't, it should return `None` as the first element of the tuple.
2. The number of visited nodes
3. The maximum size of the queue

You should use a **visited list**, as otherwise the number of explored states in this problem will be large.

<div class="alert alert-info">
Implement `depth_first_search(search_problem)` below.
</div>

In [ ]:
def depth_first_search(search_problem):
    """This function should take a PuzzleProblem instance and return a 3 element tuple as described above."""
    raise NotImplementedError()

### Solve the Puzzle Problem using DFS

Let's use your Depth First Search implementation to solve the 8 Puzzle Problem.
Execute the cell below. If your DFS implementation is correct, you should find a very long solution.

In [ ]:
# Solve the 8 Puzzle Problem from state:
# +-----+
# |  1 3|
# |4 2 5|
# |7 8 6|
# +-----+
# Don't modify this cell (contents will be overwritten by autograder)
# If you want to experiment with other states, try adding cells below.
# You can try with state: ((1, 8, 2), (0, 4, 3), (7, 6, 5)) for example.
# Remember that not all states have a solution. Try ((8, 1, 2), (0, 4, 3), (7, 6, 5)), for example.
# Be ready to wait, though!
start_state = ((0, 1, 3), (4, 2, 5), (7, 8, 6))
# start_state = ((1, 8, 2), (0, 4, 3), (7, 6, 5))
goal_state = ((1,2,3),(4,5,6),(7,8,0))
problem = PuzzleProblem(start_state, goal_state)

sol, num_visited, max_q = depth_first_search(problem)
if sol:
    print("Solution found!\n%d states in the solution (%d moves)\n%d states explored.\n%d maximum queue"
          % (len(sol.path), len(sol.path)-1, num_visited,max_q)
          )
else:
    print("No solution after exploring %d states with max q of %d" %(num_visited, max_q))

In [ ]:
# Note: the test case uses an instructor implementation of expand_nodes
Grader.run_single_test_inline(TestPSet1, "test_4_dfs", locals())

## <a name="ai_component"></a>Part 3: Differences Between BFS and DFS 

This next exercise invites you to use a generative LLM to help solve a task to find a domain that highlights the performance differences between BFS and DFS.

### <a name="llm_differentiating_example"></a>3.1 Using an LLM to Find a Differentiating Example between Performance of BFS and DFS [AI-enabled](4 points)

The goal of this exercise is to use an LLM (Claude, ChatGPT, etc.) to design a domain on which your DFS implementation will run at least _twice_ as fast (measured in number of states explored) as your BFS implementation, but your DFS implementation will return a _suboptimal_ solution (in path length) relative to your BFS implementation. The domain can be adversarial against your BFS and DFS implementation (e.g. exploit ordering in `for` loops, etc.).


<div class="alert alert-info">
Include LLM-generated code that implements the `AdversarialProblem` class, which inherits from the `PuzzleProblem` class below, to reuse your BFS and DFS implementations.
</div>

In [ ]:

class AdversarialProblem(PuzzleProblem):
    """Adversarial problem class that implements the node succession function."""
    def __init__(self, start, goal):
        super().__init__(start, goal)

        # Any persistent structure to your problem can go here.


    def expand_node(self, search_node):
        raise NotImplementedError()


def return_adversarial_problem():
    """
    Returns an instance of AdversarialProblem (with start and goal passed 
    in the constructor) to be run inside the autograder.
    """
    raise NotImplementedError()


In [ ]:
Grader.run_single_test_inline(TestPSet1, "test_5_adversarial_bfs_vs_dfs", locals())

### <a name="visualize_domain"></a>3.2 Visualize the Domain [AI-enabled] (4 points)

Next, using an LLM and visualization tools like `matplotlib` and graphical tools like `networkx` (already imported above, _please do not_ import them again), come up with a visualization of your domain to be displayed in this notebook. 
The visualization should explain why BFS was slower but optimal and DFS was faster but suboptimal in our domain.


<div class="alert alert-info">
Include the code to generate your visualization (which should appear immediately underneath the cell) below.
</div>

### <a name="time_spent"></a>4. Time Spent on Pset (5 points)

Please use [this form](https://forms.gle/QS7Raf1kvqgvCs9v5) to tell us how long you spent on this pset. After you submit the form, the form will give you a confirmation word. Please enter that confirmation word below to get an extra 5 points. 

In [ ]:
form_confirmation_word = "ENTER THE CONFIRMATION WORD HERE"

In [ ]:
# Run all tests
Grader.grade_output([TestPSet1], [locals()], "results.json")
Grader.print_test_results(
    "results.json")